# Run HRS Agent — after any dictionary/approval change

### When to run:
Run after Step 1.5.2 (approval) and 1.5.3 (coverage) complete for a use case or variables added to an existing use case's dictionary.

**What it does:** scopes the agent to the selected use case (widget below), checks the grounding landed, produces the deterministic routing report, then runs the conversational agent.

**Expected outcomes:** `COVERAGE_REPORT` = every variable has a CDM home (no deployment needed); drafts = gap variables need DDL/DML (Phase 5 review).

In [0]:
# Cell 1 — Configuration: pick the use case whose dictionary changed
dbutils.widgets.text("use_case_name", "bmi_statistics_by_wave_race_gender_age_cohort", "Use case name")
dbutils.widgets.text("codebook_version", "1992-2022v1", "Codebook version")

import os, sys, importlib, logging

# Silence Spark Connect session-policy INFO/WARNING noise on serverless
logging.getLogger("pyspark.sql.connect.logging").setLevel(logging.ERROR)

use_case = dbutils.widgets.get("use_case_name")
os.environ["HRS_AGENT_USE_CASE"] = use_case  # -> config.ACTIVE_USE_CASE_NAME

REPO_ROOT = "/Workspace/Users/peteperez.lv@gmail.com/hrs_dbx_repo"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.agent import config, hrs_tools, agent
importlib.reload(config)
print(f"ACTIVE_USE_CASE_NAME = {config.ACTIVE_USE_CASE_NAME}")

In [0]:
# Cell 2 — Grounding check: confirm the dictionary/approval rows landed for this use case
# Mirrors the agent's own filters: use_case_name + codebook_version + approval_status
version = dbutils.widgets.get("codebook_version")

approved = spark.sql(f"""
    SELECT COUNT(*) AS approved_vars,
           SUM(CASE WHEN bronze_column_match THEN 1 ELSE 0 END) AS bronze_matched
    FROM {config.APPROVED_VARIABLES_TABLE}
    WHERE use_case_name = '{use_case}'
      AND codebook_version = '{version}'
      AND approval_status = 'Approved'
""").collect()[0]

coverage = spark.sql(f"""
    SELECT SUM(CASE WHEN coverage_status = 'HAS_DATA' THEN 1 ELSE 0 END) AS has_data_rows,
           SUM(CASE WHEN coverage_status != 'HAS_DATA' THEN 1 ELSE 0 END) AS no_data_rows
    FROM {config.COHORT_COVERAGE_TABLE}
    WHERE use_case_name = '{use_case}'
      AND codebook_version = '{version}'
""").collect()[0]

print(f"Approved variables: {approved['approved_vars']} (bronze-matched: {approved['bronze_matched']})")
print(f"Coverage: {coverage['has_data_rows']} HAS_DATA / {coverage['no_data_rows']} NO_DATA")
assert approved['approved_vars'] > 0, f"No approved variables for '{use_case}' / '{version}' — did 1.5.2 run with this use_case_name?"

In [0]:
# Cell 3 — Deterministic routing report (no LLM): where do the approved variables live now?
report = hrs_tools.generate_warehouse_artifacts(use_case_name=use_case)

print(f"Status: {report['status']}")
print(f"Unrouted (gaps): {len(report.get('unrouted', []))}")
for r in report.get("routed", []):
    print(f"  {r.get('variable_pattern', r.get('variable'))} -> {r.get('cdm_table')}")
for r in report.get("report_only", []):
    print(f"  {r.get('variable')} -> report-only ({r.get('note', '')[:60]})")

In [0]:
# Cell 4 — Conversational agent run (LLM + tools, MLflow-traced, REVIEW_ONLY)
result = agent.run_agent(
    "Generate the warehouse coverage report and any silver DDL/DML drafts needed "
    "for the active use case. If every approved variable already has a CDM home, "
    "confirm with the coverage report only."
)

print(result.get("answer", ""))
print("\n--- audit ---")
for k, v in result.items():
    if k != "answer":
        print(f"{k}: {str(v)[:300]}")